# Notebook 01: Probability & Bayes Theorem

## Why This Matters

Most ML practitioners think of probability as plumbing — the loss function needs a softmax, the output needs to sum to one, done. Bayesian inference asks a more fundamental question: *what does a probability actually mean?*

The frequentist answer: a probability is the long-run frequency of an event in repeated experiments. The Bayesian answer: a probability is a *degree of belief*, updated by evidence. These aren't just philosophical positions — they lead to different tools, different answers, and different kinds of errors.

The frequentist framework dominates classical statistics and most ML training loops. But it breaks down in exactly the situations that matter most: when you can't run many experiments, when you have strong prior knowledge, when you need to quantify uncertainty rather than just point estimates, and when you're making decisions rather than just predictions.

Bayesian inference is the principled way to combine what you knew before seeing data with what the data tells you — producing a full probability distribution over hypotheses rather than a single guess.

## Background

### The Two Interpretations of Probability

**Frequentist probability**: `P(event)` = limit of (occurrences / trials) as trials → ∞. This is clean and objective, but it only applies to repeatable events. What's the frequentist probability that it rained in Paris on July 14th, 1789? There's only one such event — the frequency interpretation doesn't apply.

**Bayesian probability**: `P(hypothesis | data)` = your degree of belief in a hypothesis given evidence. This applies to any proposition you can reason about — including one-time events, model parameters, and future predictions. It's subjective (your prior can differ from mine) but disciplined: beliefs must obey the axioms of probability, and updating must follow Bayes' theorem.

The practical difference: a frequentist reports a 95% confidence interval and means "in repeated sampling, 95% of such intervals contain the true parameter." A Bayesian reports a 95% credible interval and means "given my prior and the data, there is a 95% probability the parameter lies in this interval." The Bayesian answer is what practitioners usually *want*.

### Bayes' Theorem

Bayes' theorem is a consequence of basic probability:

```
P(θ | data) = P(data | θ) × P(θ) / P(data)
```

In words:
- **Posterior** `P(θ | data)`: what we believe about θ after seeing data
- **Likelihood** `P(data | θ)`: how probable is the data if θ is true?
- **Prior** `P(θ)`: what we believed before seeing data
- **Evidence** `P(data)`: the total probability of the data under all hypotheses (normalizing constant)

The evidence is often hard to compute (it requires integrating over all possible θ), which is why MCMC and variational inference exist — they let us sample from or approximate the posterior without computing P(data) explicitly.

### The Prior — Blessing and Curse

The prior is Bayesian inference's most controversial feature. Critics say: priors are subjective — two people with different priors will reach different posteriors. Proponents say: this is a feature, not a bug — it forces you to be explicit about assumptions that frequentist analyses make implicitly.

In practice, prior choice matters most when data is scarce and least when data is abundant (the likelihood dominates). For large datasets, even very different priors converge to similar posteriors. Common prior choices:

- **Informative priors**: encode genuine domain knowledge (e.g., drug effect sizes from prior trials)
- **Weakly informative priors**: regularize without strong commitments (e.g., Normal(0, 1) on regression coefficients)
- **Non-informative / flat priors**: try to let the data speak (often improper, can cause problems)

Modern Bayesian practice strongly prefers weakly informative priors — they're more stable than flat priors while making fewer assumptions than strongly informative ones.

### Common Distributions You'll See Throughout This Series

- **Beta(α, β)**: prior for probabilities ∈ [0,1]; conjugate prior for Binomial
- **Dirichlet(α)**: prior for probability vectors; conjugate prior for Categorical
- **Normal(μ, σ)**: prior for unbounded real-valued parameters; the workhorse
- **Half-Normal / Half-Cauchy**: prior for strictly positive scale parameters (σ, τ)
- **Gamma(α, β)**: prior for rates and precision
- **Exponential(λ)**: prior for waiting times, positive parameters

## What You'll Learn

- Bayes' theorem hands-on: computing P(disease | positive test)
- The Beta distribution as a prior over probabilities
- How priors, likelihoods, and posteriors interact visually
- Working with `scipy.stats` for distributions
- The connection between Bayesian updating and conjugate priors

In [ ]:
%pip install numpy scipy matplotlib --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.gridspec as gridspec

rng = np.random.default_rng(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Ready.')

## 1. Bayes' Theorem — The Medical Test Problem

The canonical illustration. A disease has 1% prevalence. A test has 99% sensitivity (detects disease when present) and 95% specificity (correct negative when absent). You test positive. What's the probability you actually have the disease?

Intuition says ~99%. Bayes' theorem says something very different.

In [ ]:
def bayes_diagnostic(prevalence, sensitivity, specificity):
    """
    P(disease | positive test) using Bayes' theorem.

    P(disease) = prevalence (prior)
    P(+ | disease) = sensitivity (true positive rate)
    P(+ | no disease) = 1 - specificity (false positive rate)
    P(+) = P(+ | disease) * P(disease) + P(+ | no disease) * P(no disease)
    """
    p_disease = prevalence
    p_no_disease = 1 - prevalence
    p_pos_given_disease = sensitivity
    p_pos_given_no_disease = 1 - specificity

    # Total probability of positive test (marginal likelihood)
    p_pos = p_pos_given_disease * p_disease + p_pos_given_no_disease * p_no_disease

    # Bayes' theorem
    p_disease_given_pos = (p_pos_given_disease * p_disease) / p_pos
    p_no_disease_given_pos = (p_pos_given_no_disease * p_no_disease) / p_pos

    return {
        'prior_disease': p_disease,
        'p_pos': p_pos,
        'ppv': p_disease_given_pos,     # positive predictive value
        'fdr': p_no_disease_given_pos,  # false discovery rate
    }


# Standard scenario
r = bayes_diagnostic(prevalence=0.01, sensitivity=0.99, specificity=0.95)
print('Disease prevalence: 1% | Sensitivity: 99% | Specificity: 95%')
print(f'P(disease | positive): {r["ppv"]*100:.1f}%  ← probably not what you expected')
print(f'P(no disease | positive): {r["fdr"]*100:.1f}%')
print(f'P(positive): {r["p_pos"]*100:.2f}%')

print()
print('Why? Most positives are false positives:')
n = 100_000
diseased = int(n * 0.01)
healthy = n - diseased
true_pos = int(diseased * 0.99)
false_pos = int(healthy * 0.05)
print(f'  Population of {n:,}:')
print(f'  Diseased: {diseased:,}  → true positives: {true_pos:,}')
print(f'  Healthy: {healthy:,}  → false positives: {false_pos:,}')
print(f'  Of all positives ({true_pos+false_pos:,}): {true_pos/(true_pos+false_pos)*100:.1f}% are true')

print()
# Show how prevalence changes everything
print('How posterior varies with prior (prevalence):')
print(f'{"Prevalence":>12} {"P(disease | +)": >18}')
for prev in [0.001, 0.01, 0.05, 0.1, 0.5]:
    r = bayes_diagnostic(prev, 0.99, 0.95)
    print(f'{prev*100:>11.1f}% {r["ppv"]*100:>17.1f}%')

In [ ]:
# Visualize: posterior as function of prevalence (prior)
prevalences = np.linspace(0.001, 0.5, 500)
ppvs = [bayes_diagnostic(p, 0.99, 0.95)['ppv'] for p in prevalences]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(prevalences * 100, np.array(ppvs) * 100, 'b-', lw=2)
axes[0].axvline(1, color='gray', linestyle='--', alpha=0.6, label='1% prevalence')
axes[0].axvline(10, color='orange', linestyle='--', alpha=0.6, label='10% prevalence')
axes[0].set_xlabel('Prior Probability of Disease (%)')
axes[0].set_ylabel('Posterior P(disease | positive) (%)')
axes[0].set_title('How the Prior Determines the Posterior\n(Sensitivity=99%, Specificity=95%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Natural frequency diagram for 1% prevalence
axes[1].axis('off')
axes[1].set_title('Natural Frequencies (1% prevalence, 100k people)', fontsize=11)
diagram = [
    (0.5, 0.9, '100,000 people', 14, 'black'),
    (0.2, 0.7, '1,000 diseased', 12, 'darkred'),
    (0.8, 0.7, '99,000 healthy', 12, 'steelblue'),
    (0.1, 0.5, '990 true +', 10, 'red'),
    (0.3, 0.5, '10 missed', 10, 'darkred'),
    (0.7, 0.5, '4,950 false +', 10, 'tomato'),
    (0.9, 0.5, '94,050 true −', 10, 'steelblue'),
    (0.4, 0.3, 'PPV = 990 / (990+4950) = 16.7%', 12, 'purple'),
]
for x, y, text, size, color in diagram:
    axes[1].text(x, y, text, ha='center', va='center', fontsize=size, color=color)

plt.tight_layout()
plt.show()

## 2. The Beta Distribution — A Prior for Probabilities

In [ ]:
# Beta(alpha, beta) is defined on [0,1] and is the natural prior for probabilities
# alpha and beta have an intuitive interpretation:
# Think of them as "pseudo-counts" — prior successes and failures

theta = np.linspace(0, 1, 500)

configs = [
    (1, 1, 'Beta(1,1) — uniform, "no opinion"'),
    (2, 2, 'Beta(2,2) — weak belief around 0.5'),
    (5, 5, 'Beta(5,5) — moderate belief, coin is fair-ish'),
    (20, 20, 'Beta(20,20) — strong belief in fair coin'),
    (3, 10, 'Beta(3,10) — believe θ is small (biased toward tails)'),
    (10, 3, 'Beta(10,3) — believe θ is large (biased toward heads)'),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for ax, (a, b, label) in zip(axes.flat, configs):
    dist = stats.beta(a, b)
    ax.plot(theta, dist.pdf(theta), 'b-', lw=2)
    ax.fill_between(theta, dist.pdf(theta), alpha=0.2)
    ax.axvline(dist.mean(), color='red', linestyle='--', alpha=0.7,
               label=f'Mean={dist.mean():.2f}')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('θ (probability)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Beta Distribution Family — Priors for Probabilities', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Beta(α, β) interpretation:')
print('  Mean = α / (α + β)')
print('  Variance = αβ / ((α+β)²(α+β+1))')
print('  α + β ≈ strength of prior (larger = more concentrated = more confident)')
print('  Uniform(0,1) = Beta(1,1) = no prior knowledge')

## 3. Key scipy.stats Distributions — Your Toolkit

In [ ]:
# scipy.stats distributions you'll use throughout this series

def demo_distribution(dist, name, x_range, continuous=True):
    x = np.linspace(*x_range, 500)
    if continuous:
        y = dist.pdf(x)
    else:
        x = np.arange(*x_range)
        y = dist.pmf(x)
    return x, y


fig, axes = plt.subplots(2, 4, figsize=(15, 6))

distributions = [
    (stats.norm(0, 1), 'Normal(0,1)', (-4, 4), True,
     'Unbounded real params; regression coefs'),
    (stats.halfnorm(0, 1), 'HalfNormal(1)', (0, 4), True,
     'Positive scale params (σ, τ)'),
    (stats.beta(2, 5), 'Beta(2,5)', (0, 1), True,
     'Probabilities in [0,1]'),
    (stats.gamma(2, scale=1), 'Gamma(2,1)', (0, 8), True,
     'Rates, precision, positive params'),
    (stats.expon(scale=2), 'Exponential(2)', (0, 10), True,
     'Waiting times, positive params'),
    (stats.binom(20, 0.3), 'Binomial(20,0.3)', (0, 21), False,
     'Count of successes in n trials'),
    (stats.poisson(3), 'Poisson(3)', (0, 11), False,
     'Count data (events per unit)'),
    (stats.nbinom(5, 0.5), 'NegBin(5,0.5)', (0, 21), False,
     'Overdispersed count data'),
]

for ax, (dist, name, xrange, cont, desc) in zip(axes.flat, distributions):
    x, y = demo_distribution(dist, name, xrange, cont)
    if cont:
        ax.plot(x, y, 'steelblue', lw=2)
        ax.fill_between(x, y, alpha=0.2, color='steelblue')
    else:
        ax.bar(x, y, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel(desc, fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Common Bayesian Priors and Likelihoods (scipy.stats)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Demonstrate key scipy.stats API
d = stats.norm(mu := 2.0, sigma := 1.5)
print(f'Normal({mu}, {sigma}):')
print(f'  PDF at x=2.0: {d.pdf(2.0):.4f}')
print(f'  CDF at x=3.0: {d.cdf(3.0):.4f} (P(X ≤ 3))')
print(f'  PPF at 0.95:  {d.ppf(0.95):.4f} (95th percentile)')
print(f'  1000 samples: mean={d.rvs(1000, random_state=42).mean():.3f}')

## 4. Likelihood Functions — How Data Updates Beliefs

In [ ]:
# Coin flip experiment: observe n_heads heads in n_trials
# Likelihood: P(data | θ) = Binomial(n_trials, θ)

theta = np.linspace(0.001, 0.999, 500)

datasets = [
    (3, 5, 'Small data: 3/5 heads'),
    (30, 50, 'Moderate data: 30/50 heads'),
    (300, 500, 'Large data: 300/500 heads'),
    (1, 10, 'Rare events: 1/10 heads'),
]

fig, axes = plt.subplots(1, len(datasets), figsize=(14, 4))

for ax, (n_heads, n_trials, label) in zip(axes, datasets):
    # Likelihood: L(θ) = θ^k * (1-θ)^(n-k)
    log_likelihood = n_heads * np.log(theta) + (n_trials - n_heads) * np.log(1 - theta)
    likelihood = np.exp(log_likelihood - log_likelihood.max())  # normalize for display

    mle = n_heads / n_trials  # maximum likelihood estimate

    ax.plot(theta, likelihood, 'steelblue', lw=2)
    ax.fill_between(theta, likelihood, alpha=0.2, color='steelblue')
    ax.axvline(mle, color='red', linestyle='--', label=f'MLE = {mle:.2f}')
    ax.axvline(0.5, color='gray', linestyle=':', alpha=0.6, label='θ=0.5 (fair)')
    ax.set_title(label, fontsize=9)
    ax.set_xlabel('θ (coin bias)')
    ax.set_ylabel('Relative Likelihood')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Likelihood Functions: How Data Constrains θ\n(more data → narrower likelihood)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Putting It Together — Prior × Likelihood = Posterior

In [ ]:
def compute_posterior(prior_alpha, prior_beta, n_heads, n_trials):
    """
    Beta-Binomial conjugate update (analytical).

    Prior: Beta(α, β)
    Likelihood: Binomial(n, θ) with k heads
    Posterior: Beta(α + k, β + n - k)  ← exact, no sampling needed

    This is conjugacy: the prior and posterior have the same family.
    α and β can be thought of as prior pseudo-counts of heads and tails.
    """
    post_alpha = prior_alpha + n_heads
    post_beta = prior_beta + (n_trials - n_heads)
    return post_alpha, post_beta


theta = np.linspace(0.001, 0.999, 500)

# Scenario: we're trying to estimate whether a coin is biased
# We believe it's fair-ish: Beta(5, 5) prior
# We observe 7 heads in 10 flips

prior_alpha, prior_beta = 5, 5
n_heads, n_trials = 7, 10

post_alpha, post_beta = compute_posterior(prior_alpha, prior_beta, n_heads, n_trials)

prior_dist = stats.beta(prior_alpha, prior_beta)
post_dist = stats.beta(post_alpha, post_beta)
likelihood = stats.binom.pmf(n_heads, n_trials, theta)
likelihood_norm = likelihood / likelihood.max()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(theta, prior_dist.pdf(theta), 'steelblue', lw=2, label=f'Prior: Beta({prior_alpha},{prior_beta})')
ax.plot(theta, likelihood_norm * prior_dist.pdf(theta).max(), 'orange', lw=2, ls='--',
        label=f'Likelihood: {n_heads}/{n_trials} heads (scaled)')
ax.plot(theta, post_dist.pdf(theta), 'red', lw=2.5, label=f'Posterior: Beta({post_alpha},{post_beta})')

# Mark key quantities
ax.axvline(prior_dist.mean(), color='steelblue', linestyle=':', alpha=0.7,
           label=f'Prior mean: {prior_dist.mean():.2f}')
ax.axvline(n_heads/n_trials, color='orange', linestyle=':', alpha=0.7,
           label=f'MLE: {n_heads/n_trials:.2f}')
ax.axvline(post_dist.mean(), color='red', linestyle=':', alpha=0.7,
           label=f'Posterior mean: {post_dist.mean():.2f}')

ax.fill_between(theta, post_dist.pdf(theta), alpha=0.1, color='red')
ax.set_xlabel('θ (probability of heads)')
ax.set_ylabel('Density / Relative Likelihood')
ax.set_title('Prior × Likelihood → Posterior\n'
             f'Prior: Beta({prior_alpha},{prior_beta}), '
             f'Data: {n_heads}/{n_trials} heads, '
             f'Posterior: Beta({post_alpha},{post_beta})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Credible interval
ci_low, ci_high = post_dist.ppf(0.025), post_dist.ppf(0.975)
print(f'Posterior: Beta({post_alpha}, {post_beta})')
print(f'Posterior mean: {post_dist.mean():.4f}')
print(f'95% credible interval: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'Interpretation: there is a 95% probability θ lies in [{ci_low:.2f}, {ci_high:.2f}]')
print()
print('The posterior mean is a compromise between the prior mean and the MLE:')
print(f'  Prior weight: {prior_alpha + prior_beta} pseudo-counts')
print(f'  Data weight:  {n_trials} actual observations')
print(f'  As n → ∞, posterior converges to MLE regardless of prior')

## 6. Sequential Updating — Bayesian Inference as an Online Process

In [ ]:
# One of the most useful properties of Bayesian inference:
# today's posterior becomes tomorrow's prior.
# This makes it naturally sequential.

# Simulate a coin with true bias θ=0.65
true_theta = 0.65
n_flips = 50
flips = rng.binomial(1, true_theta, n_flips)

# Start with uniform prior, update flip by flip
alpha, beta_param = 1, 1  # Beta(1,1) = Uniform
theta_grid = np.linspace(0.001, 0.999, 500)

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
checkpoints = [1, 3, 10, 20, 35, 50]

a_cur, b_cur = alpha, beta_param
results = []

for i, flip in enumerate(flips):
    a_cur += flip
    b_cur += (1 - flip)
    if (i + 1) in checkpoints:
        results.append((i + 1, a_cur, b_cur))

for ax, (n, a, b) in zip(axes.flat, results):
    dist = stats.beta(a, b)
    ax.plot(theta_grid, dist.pdf(theta_grid), 'steelblue', lw=2)
    ax.fill_between(theta_grid, dist.pdf(theta_grid), alpha=0.2, color='steelblue')
    ax.axvline(true_theta, color='red', linestyle='--', alpha=0.8, label=f'True θ={true_theta}')
    ax.axvline(dist.mean(), color='green', linestyle=':', alpha=0.8,
               label=f'Post. mean={dist.mean():.3f}')
    ax.set_title(f'After {n} flips ({a-1} heads, {b-1} tails)', fontsize=9)
    ax.set_xlabel('θ')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Sequential Bayesian Updating — Prior → Posterior After Each Flip', fontsize=12)
plt.tight_layout()
plt.show()

print(f'True bias: {true_theta}')
print(f'Final posterior: Beta({a_cur}, {b_cur})')
final_dist = stats.beta(a_cur, b_cur)
print(f'Final posterior mean: {final_dist.mean():.4f}')
print(f'Final 95% CI: [{final_dist.ppf(0.025):.3f}, {final_dist.ppf(0.975):.3f}]')
print(f'True θ in 95% CI: {final_dist.ppf(0.025) <= true_theta <= final_dist.ppf(0.975)}')

## 7. Common Distributions Toolkit

In [ ]:
# Quick reference: when to use each distribution as a prior

scenarios = [
    ('Conversion rate', 'Beta(1,1) — uniform, or Beta(2,10) if rate is expected small'),
    ('Regression coefficient', 'Normal(0, 1) — weakly informative; most real effects are small'),
    ('Standard deviation / scale', 'HalfNormal(1) or HalfCauchy(2.5) — positive, not too large'),
    ('Count of events', 'Poisson(λ) likelihood; Gamma prior on λ'),
    ('Time to event', 'Exponential(λ) or Weibull likelihood; Gamma prior on λ'),
    ('Probability vector', 'Dirichlet(α) — generalization of Beta to K categories'),
    ('Treatment effect (bounded)', 'Normal(0, 0.5) — small effects are more common than large'),
    ('Heavy-tailed noise', 'StudentT(ν=3) likelihood — robust to outliers'),
]

print('Prior Selection Reference:')
print('='*70)
for scenario, rec in scenarios:
    print(f'  {scenario}:')
    print(f'    → {rec}')
    print()

# Demonstrate Dirichlet
dirichlet_configs = [
    ([1, 1, 1, 1], 'Dirichlet([1,1,1,1]) — uniform over simplex'),
    ([0.1, 0.1, 0.1, 0.1], 'Dirichlet([0.1,0.1,0.1,0.1]) — sparse (one category dominates)'),
    ([10, 10, 10, 10], 'Dirichlet([10,10,10,10]) — concentrated around equal proportions'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (alpha_vec, label) in zip(axes, dirichlet_configs):
    # Sample from Dirichlet and plot category distributions
    samples = rng.dirichlet(alpha_vec, size=1000)
    ax.boxplot(samples, labels=[f'Cat {i+1}' for i in range(4)])
    ax.set_title(label, fontsize=9)
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Dirichlet Distribution — Generalizing Beta to K Categories', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

### Key Concepts

| Concept | Formula | Interpretation |
|---------|---------|---------------|
| Bayes' theorem | `P(θ|data) = P(data|θ) P(θ) / P(data)` | Prior belief + evidence → updated belief |
| Prior | `P(θ)` | What you believe before seeing data |
| Likelihood | `P(data|θ)` | How probable is the data given θ? |
| Posterior | `P(θ|data)` | Updated belief — the answer we want |
| Credible interval | `[P.025, P.975]` of posterior | 95% probability that θ lies here |
| Beta-Binomial | `Beta(α+k, β+n-k)` | Conjugate update for coin-flip problems |

### The Bayesian Workflow

1. **Choose a prior** `P(θ)` — encode domain knowledge, use weakly informative defaults when in doubt
2. **Specify a likelihood** `P(data|θ)` — what data-generating process fits your problem?
3. **Compute the posterior** `P(θ|data)` — analytically (conjugate) or numerically (MCMC, VI)
4. **Interpret the posterior** — mean, credible intervals, predictive distributions
5. **Check the model** — posterior predictive checks (notebook 11)

### What's Coming

- **Notebook 02**: The conjugate update loop — Beta-Binomial, Normal-Normal, Poisson-Gamma in detail
- **Notebook 03**: MCMC — when conjugacy isn't available and we need to sample from the posterior numerically
- **Notebook 04**: PyMC — the Python library that makes all of this practical for real models

Next: Notebook 02 dives into the Bayesian update loop — implementing conjugate models from scratch, comparing them to the general MCMC approach, and building intuition for what "inference" means computationally.